<a href="https://colab.research.google.com/github/everestso/Atari_2600_Games/blob/main/s264s26_MsPacman.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U stable-baselines3 huggingface_hub "gymnasium[atari,accept-rom-license]" ale-py shimmy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 11.2 MB/s eta 0:00:00


In [2]:
import gymnasium as gym
import ale_py

gym.register_envs(ale_py)

matches = sorted([env for env in gym.registry.keys() if "Pacman" in env or "MsPacman" in env])
print(matches)

['ALE/MsPacman-v5', 'ALE/Pacman-v5', 'MsPacman-v0', 'MsPacman-v4', 'MsPacmanNoFrameskip-v0', 'MsPacmanNoFrameskip-v4']


In [3]:
# CELL 1
import os

from stable_baselines3 import A2C
from huggingface_hub import hf_hub_download

repo_id = "sb3/a2c-MsPacmanNoFrameskip-v4"
filename = "a2c-MsPacmanNoFrameskip-v4.zip"

model_path = hf_hub_download(
    repo_id=repo_id,
    filename=filename,
    token=False
)

if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model file not found: {model_path}")

model = A2C.load(
    model_path,
    custom_objects={
        "learning_rate": 0.0,
        "lr_schedule": lambda _: 0.0,
        "clip_range": lambda _: 0.0,
    }
)

print("Loaded model:", model_path)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


a2c-MsPacmanNoFrameskip-v4.zip:   0%|          | 0.00/13.7M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/save_util.py:165: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  deserialized_object = cloudpickle.loads(base64_object)
/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/vec_env/patch_gym.py:95: UserWarning: You loaded a model that was trained using OpenAI Gym. We strongly recommend transitioning to Gymnasium by saving that model again.
  warnings.warn(


Loaded model: /root/.cache/huggingface/hub/models--sb3--a2c-MsPacmanNoFrameskip-v4/snapshots/8d741f2b9ec1107e7284f892d058a619da6e328f/a2c-MsPacmanNoFrameskip-v4.zip


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/base_class.py:751: UserWarning: You are probably loading a A2C/PPO model saved with SB3 < 1.7.0, we deactivated exact_match so you can save the model again to avoid issues in the future (see https://github.com/DLR-RM/stable-baselines3/issues/1233 for more info). Original error: Error(s) in loading state_dict for ActorCriticCnnPolicy:
	Missing key(s) in state_dict: "pi_features_extractor.cnn.0.weight", "pi_features_extractor.cnn.0.bias", "pi_features_extractor.cnn.2.weight", "pi_features_extractor.cnn.2.bias", "pi_features_extractor.cnn.4.weight", "pi_features_extractor.cnn.4.bias", "pi_features_extractor.linear.0.weight", "pi_features_extractor.linear.0.bias", "vf_features_extractor.cnn.0.weight", "vf_features_extractor.cnn.0.bias", "vf_features_extractor.cnn.2.weight", "vf_features_extractor.cnn.2.bias", "vf_features_extractor.cnn.4.weight", "vf_features_extractor.cnn.4.bias", "vf_features_extractor.linear.0.weight", "vf

In [4]:
# CELL 2
import os
import glob
import time
import base64
from IPython.display import HTML, display

import gymnasium as gym
import ale_py

gym.register_envs(ale_py)

from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack, VecVideoRecorder

video_folder = "videos"
os.makedirs(video_folder, exist_ok=True)

for f in glob.glob(os.path.join(video_folder, "*.mp4")):
    os.remove(f)

env_id = "MsPacmanNoFrameskip-v4"
print("Using environment:", env_id)

eval_env = make_atari_env(
    env_id,
    n_envs=1,
    seed=0,
    wrapper_kwargs=dict(terminal_on_life_loss=False),
)
eval_env = VecFrameStack(eval_env, n_stack=4)

video_length = 2000

eval_env = VecVideoRecorder(
    eval_env,
    video_folder=video_folder,
    record_video_trigger=lambda step: step == 0,
    video_length=video_length,
    name_prefix="a2c-mspacman"
)

obs = eval_env.reset()

max_seconds = 60
start_time = time.time()

step_count = 0
done = [False]
episode_reward = 0.0

while (
    step_count < video_length
    and (time.time() - start_time) < max_seconds
    and not done[0]
):
    action, _states = model.predict(obs, deterministic=True)
    obs, rewards, done, infos = eval_env.step(action)
    episode_reward += float(rewards[0])
    step_count += 1

eval_env.close()

elapsed = time.time() - start_time
print(f"Stopped after {step_count} steps and {elapsed:.1f} seconds.")
print(f"Approx reward: {episode_reward:.2f}")

video_files = glob.glob(os.path.join(video_folder, "*.mp4"))
if video_files:
    video_path = sorted(video_files)[-1]
    print("Video saved to:", video_path)

    with open(video_path, "rb") as f:
        mp4 = f.read()
    data_url = "data:video/mp4;base64," + base64.b64encode(mp4).decode()

    display(HTML(f"""
    <div style="width: 100%; overflow-x: auto;">
        <video controls playsinline style="width: 100%; max-width: 360px; height: auto;">
            <source src="{data_url}" type="video/mp4">
        </video>
    </div>
    """))
else:
    print("No video file was created.")

Using environment: MsPacmanNoFrameskip-v4


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"


Moviepy - Building video /content/videos/a2c-mspacman-step-0-to-step-2000.mp4.
Moviepy - Writing video /content/videos/a2c-mspacman-step-0-to-step-2000.mp4



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Moviepy - Done !
Moviepy - video ready /content/videos/a2c-mspacman-step-0-to-step-2000.mp4
Stopped after 769 steps and 5.3 seconds.
Approx reward: 118.00
Video saved to: videos/a2c-mspacman-step-0-to-step-2000.mp4
